<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>

<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
  text-align: center;
}
</style>

# Docker Security and Duckiedrone Boundaries

This notebook explains Docker security boundaries between local practice, physical Duckiedrones, remote Docker daemons, and host resources.

## Security boundaries and safe defaults

Docker isolates processes, but a container is only as restricted as the access it receives when it starts. Follow the principle of least privilege: give a container only the files, devices, network access, and permissions required for its specific task.

Treat access to the Docker daemon as powerful administrative access to the base station. A person or container that can ask the daemon to start privileged containers, use host devices, or mount host files can potentially affect the host. In particular, never mount `/var/run/docker.sock` into a practice container. Do not use `--privileged`, `--device`, host namespaces such as `--network host`, or added Linux capabilities unless an authorized deployment specifically requires them and documents their purpose.

The Docker API can also be exposed over Transmission Control Protocol (TCP). By convention, port `2375` is used for an unencrypted Docker API, while port `2376` is used for an API configured with Transport Layer Security (TLS). Port numbers do not provide protection by themselves: access to either API is powerful host-level access. Port `2376` is appropriate only when TLS, client-certificate verification, and network access controls are configured correctly. This learning experience does not ask learners to configure a TCP Docker API or set `DOCKER_HOST` to a TCP address. An authorized Duckietown deployment workflow can manage a remote Duckiedrone endpoint, but it is distinct from local Docker practice in this learning experience. Do not expose or reconfigure a Docker daemon for this learning experience.

Mount only the supplied exercise path when a container needs files. Prefer a named volume for container-owned data and a read-only bind mount when the container only needs to inspect host files. Do not mount home directories, credentials, keys, or other sensitive host paths. Do not copy secrets into a Dockerfile, an image, or its build context; use `.dockerignore` to keep unnecessary and sensitive files out of the build context.

Use images from sources you trust and inspect unfamiliar image names before running them. An explicit tag improves repeatability, but it can still be changed by its publisher; use a supplied digest when exact image contents matter. The local image workflow applies these defaults by running as a non-root account, publishing the practice server only on `127.0.0.1`, and using a read-only bind mount.

## Docker on the Duckiedrone

A physical Duckiedrone runs Docker on its own embedded computer. Its platform services run in containers that are configured by Duckietown tooling. A virtual Duckiedrone is also implemented with containers, but it runs on the base station. Neither case makes it safe to experiment with arbitrary Docker commands on platform containers.

Keep the command context clear. [Table 1](#table-1) distinguishes the normal Docker target in each context.

<table id="table-1" class="lx-table">
  <caption>Table 1: Normal Docker targets by command context.</caption>
  <thead>
    <tr>
      <th>Where the command runs</th>
      <th>What the local Docker daemon normally manages</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Authorized base-station terminal</td>
      <td>Learner-owned practice containers on the base station</td>
    </tr>
    <tr>
      <td>Physical Duckiedrone shell</td>
      <td>Duckiedrone platform containers on that Duckiedrone</td>
    </tr>
    <tr>
      <td>Shell inside a container</td>
      <td>Usually no Docker daemon unless one was deliberately provided</td>
    </tr>
  </tbody>
</table>

The practical notebooks use only clearly named practice resources on the base station. The networking LX explains why `localhost` inside a container normally refers to that container, not automatically to the base station or a physical Duckiedrone.

## Further reading

See the [Duckiedrone DD24 manual](https://docs.duckietown.com/ente/opmanual-dd24/) for the Duckiedrone operating context.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()
